#  Wildfire Detection Using Deep Learning

**Objective:** Develop and compare deep learning models for automated wildfire detection from satellite imagery, enhanced by systematic preprocessing pipeline optimization.

**Approach:**
1. **Phase 1 — Preprocessing Ablation:** Compare 4 preprocessing pipelines using a lightweight CNN to select the optimal strategy.
2. **Phase 2 — Transfer Learning:** Train VGG16, ResNet50, EfficientNetB0, and a Custom CNN using the best pipeline with equal fine-tuning for all.
3. **Phase 3 — Evaluation & Explainability:** F1-optimal threshold, ensemble, error analysis, and Grad-CAM visualization.

**Dataset:** [Wildfire Prediction Dataset](https://www.kaggle.com/datasets/abdelghaniaaba/wildfire-prediction-dataset) via KaggleHub

---

## Table of Contents
1. Environment Setup & Dataset Download
2. Exploratory Data Analysis
3. Preprocessing Pipeline Definitions
4. Preprocessing Ablation Study
5. Transfer Learning Model Architecture
6. Training All Models (64 epochs, resumable)
7. Fine-tuning All Models Equally
8. Ensemble Model
9. Evaluation & Comparison
10. F1-Optimal Threshold (not default 0.5)
11. Error Analysis
12. Grad-CAM Visualization
13. Conclusions & Future Work


## 1. Environment Setup & Dataset Download

In [ ]:
# ── Core imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import json
import pickle
import shutil
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

# ── Deep Learning ──────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D, BatchNormalization, Input, Activation
)
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

# ── Metrics ───────────────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    average_precision_score, precision_recall_curve
)
from sklearn.calibration import calibration_curve

# ── Reproducibility ───────────────────────────────────────────────────────────
import random
SEED = 42
random.seed(SEED)           # Python built-in
np.random.seed(SEED)        # NumPy
os.environ['PYTHONHASHSEED'] = str(SEED)  # hash-based ops
tf.random.set_seed(SEED)    # TensorFlow / Keras

# ── Plot style ────────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# ── Colour palette (one colour per model, consistent across all plots) ─────────
pal = {
    'VGG16':          '#e74c3c',
    'ResNet50':       '#3498db',
    'EfficientNetB0': '#9b59b6',
    'CustomCNN':      '#2ecc71',
    'Ensemble':       '#f39c12',
}

print('TensorFlow version :', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU Available      :', gpus)
if not gpus:
    print('\n⚠  No GPU detected — training will be slow on CPU.')


In [ ]:
import kagglehub

path = kagglehub.dataset_download('abdelghaniaaba/wildfire-prediction-dataset')
print('Path to dataset files:', path)

# ── Directory Setup ────────────────────────────────────────────────────────────
BASE_DIR        = Path(path)
TRAIN_DIR       = BASE_DIR / 'train'
VALID_DIR       = BASE_DIR / 'valid'
TEST_DIR        = BASE_DIR / 'test'
OUT_DIR         = Path('outputs')
SAVE_DIR        = OUT_DIR / 'saved_models'
CHECKPOINT_DIR  = OUT_DIR / 'checkpoints'
for d in [OUT_DIR, SAVE_DIR, CHECKPOINT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Global Configuration ───────────────────────────────────────────────────────
IMG_SIZE     = 224          # Standard size for ImageNet pretrained models
BATCH_SIZE   = 64
EPOCHS       = 64           # Max epochs — EarlyStopping will trigger earlier
CLASS_NAMES  = ['nowildfire', 'wildfire']
NUM_CLASSES  = len(CLASS_NAMES)
WILDFIRE_IDX = 1            # Positive class index

print(f'Image size  : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size  : {BATCH_SIZE}')
print(f'Max epochs  : {EPOCHS}')
print(f'Classes     : {CLASS_NAMES}')


## 2. Exploratory Data Analysis

Before training, we explore the dataset to understand:
- Class distribution across splits
- Visual characteristics of each class
- Potential class imbalance issues


In [ ]:
def count_images(directory):
    counts = {}
    exts = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG')
    for cls in CLASS_NAMES:
        cls_path = directory / cls
        if cls_path.exists():
            counts[cls] = sum(len(list(cls_path.glob(e))) for e in exts)
        else:
            counts[cls] = 0
    return counts

train_counts = count_images(TRAIN_DIR)
valid_counts = count_images(VALID_DIR)
test_counts  = count_images(TEST_DIR)

print('=' * 60)
print('DATASET STATISTICS')
print('=' * 60)
for split_name, counts in [('Train', train_counts), ('Valid', valid_counts), ('Test', test_counts)]:
    print(f'\n{split_name} Set:')
    for cls, n in counts.items():
        print(f'  {cls:15s}: {n:5d} images')
    print(f'  {"Total":15s}: {sum(counts.values()):5d} images')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ['#3498db', '#e74c3c']
for ax, (split_name, counts) in zip(axes, [
    ('Training Set', train_counts),
    ('Validation Set', valid_counts),
    ('Test Set', test_counts)
]):
    vals = list(counts.values())
    bars = ax.bar(CLASS_NAMES, vals, color=colors, edgecolor='white', linewidth=2)
    ax.set_title(split_name, fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of Images')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + max(vals)*0.02,
                str(val), ha='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
def display_sample_images(directory, num_samples=5):
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
    fig.suptitle('Sample Images from Dataset', fontsize=14, fontweight='bold')
    for row, cls in enumerate(CLASS_NAMES):
        cls_path = directory / cls
        img_files = list(cls_path.glob('*.jpg'))[:num_samples]
        for col, img_path in enumerate(img_files):
            img = cv2.imread(str(img_path))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(img)
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(cls.upper(), fontsize=11,
                    fontweight='bold', rotation=0, ha='right', va='center')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'sample_images.png', dpi=150, bbox_inches='tight')
    plt.show()

display_sample_images(TRAIN_DIR, num_samples=5)


## 3. Image Preprocessing Pipelines

### Why Preprocessing Matters

Raw satellite imagery often suffers from low contrast in smoky conditions, color degradation, and noise. We test **4 distinct pipelines** to find which best reveals fire-related features.

| Technique | Description |
|-----------|-------------|
| **CLAHE** | Adaptive histogram equalization on the L channel (LAB space) — boosts local contrast without over-brightening |
| **Fire Color Enhancement** | Increases HSV saturation and brightness to highlight red/orange fire regions |
| **Gaussian Smoothing** | Light 3×3 blur to reduce noise — applied last to preserve edges |

> **Note on EfficientNetB0 preprocessing:** EfficientNetB0 has a built-in rescaling layer that expects raw `[0, 255]` uint8 pixels.
> Applying an external normalisation pipeline on top causes a double-rescaling bug (values collapse to ~0).
> Fix: `include_preprocessing=False` is passed so our pipeline controls normalisation for all models uniformly.


In [ ]:
# ── Preprocessing functions ───────────────────────────────────────────────────
def apply_clahe(img_bgr, clip_limit=2.0, tile_grid=(8, 8)):
    """CLAHE on L channel of LAB space — preserves colour, boosts local contrast."""
    img_bgr = img_bgr.astype(np.uint8)
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    l = clahe.apply(l.astype(np.uint8))
    lab = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def enhance_fire_colors(img_bgr, saturation_boost=1.3, value_boost=1.2):
    """Boost HSV saturation + brightness to make fire regions more prominent."""
    img_bgr = img_bgr.astype(np.uint8)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    h, s, v = cv2.split(hsv)
    s = np.clip(s * saturation_boost, 0, 255)
    v = np.clip(v * value_boost,      0, 255)
    hsv = cv2.merge([h, s, v]).astype(np.uint8)
    return cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)

def apply_gaussian_smoothing(img_bgr, kernel_size=(3, 3)):
    """Light Gaussian blur for noise reduction. Small kernel preserves edges."""
    return cv2.GaussianBlur(img_bgr.astype(np.uint8), kernel_size, 0)

def pipeline_clahe(img_bgr):
    img = cv2.resize(img_bgr.astype(np.uint8), (IMG_SIZE, IMG_SIZE))
    return apply_clahe(img).astype(np.float32)

def pipeline_fire_enhancement(img_bgr):
    img = cv2.resize(img_bgr.astype(np.uint8), (IMG_SIZE, IMG_SIZE))
    return enhance_fire_colors(img).astype(np.float32)

def pipeline_clahe_fire(img_bgr):
    """WINNER from ablation study — CLAHE then fire colour enhancement."""
    img = cv2.resize(img_bgr.astype(np.uint8), (IMG_SIZE, IMG_SIZE))
    img = apply_clahe(img)
    return enhance_fire_colors(img).astype(np.float32)

def pipeline_full(img_bgr):
    img = cv2.resize(img_bgr.astype(np.uint8), (IMG_SIZE, IMG_SIZE))
    img = apply_clahe(img)
    img = enhance_fire_colors(img)
    return apply_gaussian_smoothing(img).astype(np.float32)

PIPELINES = {
    'CLAHE':                 pipeline_clahe,
    'Fire Enhancement':      pipeline_fire_enhancement,
    'CLAHE + Fire':          pipeline_clahe_fire,
    'CLAHE + Fire + Smooth': pipeline_full,
}

print('Preprocessing pipelines registered:')
for name in PIPELINES:
    print(f'  ✓  {name}')


In [ ]:
def visualize_pipelines():
    sample_fire   = next((TRAIN_DIR / 'wildfire').glob('*.jpg'), None)
    sample_nofire = next((TRAIN_DIR / 'nowildfire').glob('*.jpg'), None)
    samples = [(sample_fire, 'Wildfire'), (sample_nofire, 'No Wildfire')]
    n_pipes = len(PIPELINES)
    fig, axes = plt.subplots(2, n_pipes + 1, figsize=(4 * (n_pipes + 1), 6))
    fig.suptitle('Preprocessing Pipeline Effects on Sample Images',
                 fontsize=14, fontweight='bold')
    for row, (img_path, label) in enumerate(samples):
        img_bgr = cv2.imread(str(img_path))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        axes[row, 0].imshow(img_rgb)
        axes[row, 0].set_title('Original', fontsize=9, fontweight='bold')
        axes[row, 0].set_ylabel(label, fontsize=11, fontweight='bold')
        axes[row, 0].axis('off')
        for col, (name, fn) in enumerate(PIPELINES.items(), start=1):
            processed = fn(img_bgr.copy()).astype(np.uint8)
            processed_rgb = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)
            axes[row, col].imshow(processed_rgb)
            axes[row, col].set_title(name, fontsize=8)
            axes[row, col].axis('off')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'preprocessing_visual_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_pipelines()


## 4. Preprocessing Ablation Study

Train a lightweight CNN for 7 epochs on each pipeline to pick the winner without running all 4 full training jobs.

### Ablation CNN Architecture
```
Conv2D(32) → MaxPool → Conv2D(64) → MaxPool → GlobalAvgPool → Dense(64) → Dropout(0.4) → Sigmoid
```


In [ ]:
def build_ablation_cnn():
    """Small fast CNN used only to rank preprocessing pipelines."""
    model = Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.MaxPooling2D(2),
        layers.GlobalAveragePooling2D(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.4),
        layers.Dense(1, activation='sigmoid'),
    ], name='Ablation_CNN')
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

build_ablation_cnn().summary()


In [ ]:
def make_pipeline_generator(split_dir, pipeline_fn, batch_size=BATCH_SIZE, shuffle=True):
    """
    Wraps a preprocessing pipeline into a Keras ImageDataGenerator.
    The pipeline receives a BGR uint8 image and returns float32 BGR in [0,255].
    We normalise to [0,1] here.
    """
    def preprocess(img_rgb):
        img_rgb = img_rgb.astype(np.uint8)
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        processed = pipeline_fn(img_bgr)
        return np.clip(processed, 0, 255).astype(np.float32) / 255.0
    gen = ImageDataGenerator(preprocessing_function=preprocess)
    return gen.flow_from_directory(
        split_dir, target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=batch_size, class_mode='binary', shuffle=shuffle, seed=SEED
    )


In [ ]:
print('=' * 70)
print('PREPROCESSING ABLATION STUDY  (4 pipelines × 7 epochs each)')
print('=' * 70)

ablation_results   = {}
ablation_histories = {}
ablation_models    = {}

for pipe_name, pipe_fn in PIPELINES.items():
    print(f'\n--- Pipeline: {pipe_name} ---')
    tr = make_pipeline_generator(TRAIN_DIR, pipe_fn, shuffle=True)
    vl = make_pipeline_generator(VALID_DIR, pipe_fn, shuffle=False)
    te = make_pipeline_generator(TEST_DIR,  pipe_fn, shuffle=False)
    m  = build_ablation_cnn()
    cb = [
        callbacks.EarlyStopping(monitor='val_accuracy', patience=3,
                                restore_best_weights=True, verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                    patience=2, min_lr=1e-7, verbose=0)
    ]
    hist = m.fit(tr, epochs=7, validation_data=vl, callbacks=cb, verbose=1)
    te.reset()
    loss, acc, auc = m.evaluate(te, verbose=0)
    ablation_results[pipe_name]   = {
        'test_accuracy': acc, 'test_auc': auc,
        'best_val_acc': max(hist.history['val_accuracy']),
        'best_val_auc': max(hist.history['val_auc']),
    }
    ablation_histories[pipe_name] = hist
    ablation_models[pipe_name]    = m
    print(f'  Test Acc: {acc:.4f}  |  Test AUC: {auc:.4f}')

print('\n' + '=' * 70)
print('ABLATION COMPLETE')
print('=' * 70)


In [ ]:
ablation_df = pd.DataFrame([
    {'Pipeline': k, **v} for k, v in ablation_results.items()
]).sort_values('test_accuracy', ascending=False)

print(ablation_df.to_string(index=False, float_format='{:.4f}'.format))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Preprocessing Ablation Results', fontsize=14, fontweight='bold')

ax = axes[0]
pipe_names = ablation_df['Pipeline'].tolist()
test_accs  = ablation_df['test_accuracy'].tolist()
color_abl  = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(pipe_names))]
bars = ax.barh(pipe_names, test_accs, color=color_abl, edgecolor='white', linewidth=1.5)
bars[0].set_edgecolor('gold'); bars[0].set_linewidth(3)
ax.set_xlabel('Test Accuracy', fontsize=11, fontweight='bold')
ax.set_title('Test Accuracy by Pipeline', fontsize=12, fontweight='bold')
ax.set_xlim([0.7, 1.0])
for bar, val in zip(bars, test_accs):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9, fontweight='bold')

ax = axes[1]
colors_line = ['#95a5a6', '#e74c3c', '#f39c12', '#3498db']
for (pipe_name, hist), color in zip(ablation_histories.items(), colors_line):
    ep = range(1, len(hist.history['val_accuracy']) + 1)
    ax.plot(ep, hist.history['val_accuracy'], label=pipe_name, color=color, lw=2)
ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax.set_ylabel('Validation Accuracy', fontsize=11, fontweight='bold')
ax.set_title('Val Accuracy During Ablation', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.savefig(OUT_DIR / 'ablation_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Select best pipeline ───────────────────────────────────────────────────────
BEST_PIPELINE_NAME = ablation_df.iloc[0]['Pipeline']
BEST_PIPELINE_FN   = PIPELINES[BEST_PIPELINE_NAME]
print(f'\n  BEST PIPELINE : {BEST_PIPELINE_NAME}')
print(f'    Test Accuracy  : {ablation_df.iloc[0]["test_accuracy"]:.4f}')
print('\n→  This pipeline will be applied to ALL models.')


In [ ]:
print('Saving ablation models and results...')
for pipe_name, model in ablation_models.items():
    model.save(SAVE_DIR / f'ablation_{pipe_name.replace(" ", "_")}.keras')
    print(f'  ✔ {pipe_name}')
ablation_df.to_csv(SAVE_DIR / 'ablation_results.csv', index=False)
with open(SAVE_DIR / 'ablation_results.json', 'w') as f:
    json.dump(ablation_results, f, indent=4)
with open(SAVE_DIR / 'best_pipeline_info.json', 'w') as f:
    json.dump({'best_pipeline_name': BEST_PIPELINE_NAME,
               'best_test_accuracy': float(ablation_df.iloc[0]['test_accuracy']),
               'best_test_auc':      float(ablation_df.iloc[0]['test_auc'])}, f, indent=4)
print('All ablation outputs saved.')


## 5. Transfer Learning Model Architecture

We attach a binary classification head to each pre-trained base. All bases are frozen initially; all will be fine-tuned equally in Phase 2.

### Architecture (shared head)
```
[Pre-trained Base (frozen)] → GlobalAveragePooling2D
    → Dense(256, ReLU) → BatchNorm → Dropout(0.5) → Dense(1, Sigmoid)
```

### FIX — EfficientNetB0 preprocessing
`include_preprocessing=False` disables EfficientNetB0's built-in rescaling layer.
This lets our CLAHE+Fire pipeline control normalisation uniformly for all models,
eliminating the double-rescaling bug that caused the previous 85% AUC.


In [ ]:
def _wrap_pipeline(pipeline_fn):
    """Convert pipeline_fn (BGR float32 → float32) into a Keras preprocessing_function (RGB uint8 → float32 [0,1])."""
    def preprocess(img_rgb):
        img_rgb = img_rgb.astype(np.uint8)
        img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
        processed = pipeline_fn(img_bgr)
        return np.clip(processed, 0, 255).astype(np.float32) / 255.0
    return preprocess

_prep_fn = _wrap_pipeline(BEST_PIPELINE_FN)

# Training generators — satellite-specific augmentation
# brightness_range: simulates different lighting / sensor gain across satellites
# channel_shift_range: mimics spectral variability between sensor types (Sentinel-2, Landsat)
train_datagen = ImageDataGenerator(
    preprocessing_function=_prep_fn,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    fill_mode='nearest',
    brightness_range=[0.8, 1.2],   # ← sensor gain / sunlight variation
    channel_shift_range=15.0        # ← spectral variability across satellite sensors
)

# Validation / test — pipeline only, no augmentation
eval_datagen = ImageDataGenerator(preprocessing_function=_prep_fn)

def make_generators(model_name):
    """Return fresh (train, val, test) generators for a given model name."""    common_kw = dict(target_size=(IMG_SIZE, IMG_SIZE),
                     batch_size=BATCH_SIZE, class_mode='binary', seed=SEED)
    tr = train_datagen.flow_from_directory(TRAIN_DIR, shuffle=True,  **common_kw)
    vl = eval_datagen .flow_from_directory(VALID_DIR, shuffle=False, **common_kw)
    te = eval_datagen .flow_from_directory(TEST_DIR,  shuffle=False, **common_kw)
    return tr, vl, te

print(f'Generators ready with pipeline: "{BEST_PIPELINE_NAME}"')


## 6. Build Models & Callbacks

All models share the same binary head and identical callback configuration.

In [ ]:
# ── FIX: build_transfer_model — binary throughout (sigmoid + binary_crossentropy) ──
def build_transfer_model(base_model, model_name):
    """Attach a frozen pre-trained base to a binary classification head."""
    for layer in base_model.layers:
        layer.trainable = False

    model = Sequential([
        base_model,
        GlobalAveragePooling2D(name='gap'),
        Dense(256, activation='relu', name='dense_256'),
        BatchNormalization(name='bn'),
        Dropout(0.5, name='dropout'),
        Dense(1, activation='sigmoid', name='output'),   # ← binary output
    ], name=model_name)

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',                      # ← binary loss
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model


# ── FIX: EfficientNetB0 — include_preprocessing=False to avoid double rescaling ──
base_cfg      = dict(include_top=False, weights='imagenet',
                     input_shape=(IMG_SIZE, IMG_SIZE, 3))
eff_base_cfg  = dict(include_top=False, weights='imagenet',
                     input_shape=(IMG_SIZE, IMG_SIZE, 3),
                     include_preprocessing=False)        # ← disables built-in rescaling

MODELS_TO_TRAIN = {
    'VGG16':          build_transfer_model(VGG16(**base_cfg),                  'VGG16_WF'),
    'ResNet50':       build_transfer_model(ResNet50(**base_cfg),               'ResNet50_WF'),
    'EfficientNetB0': build_transfer_model(EfficientNetB0(**eff_base_cfg),     'EfficientNetB0_WF'),
}

# ── Custom CNN (as specified) ─────────────────────────────────────────────────
def build_custom_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    """
    Custom lightweight CNN with 3 blocks.
    3x [Conv -> BatchNorm -> ReLU -> MaxPool]
    -> GlobalAveragePool -> Dense(256) -> Dropout(0.5) -> sigmoid
    """
    inp = layers.Input(shape=input_shape, name='input')
    x = inp
    for filters, name in [(32, '1'), (64, '2'), (128, '3')]:
        x = layers.Conv2D(filters, 3, padding='same', activation=None, name=f'conv{name}')(x)
        x = layers.BatchNormalization(name=f'bn{name}')(x)
        x = layers.Activation('relu', name=f'relu{name}')(x)
        if name != '3':
            # Block 3 intentionally skips MaxPool — keeping spatial resolution
            # higher before GlobalAveragePooling2D improves fine-grained feature
            # retention for small fire regions without adding extra parameters.
            x = layers.MaxPooling2D(2, name=f'pool{name}')(x)
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5, name='dropout')(x)
    out = layers.Dense(1, activation='sigmoid', name='output')(x)
    model = models.Model(inp, out, name='CustomCNN')
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model

MODELS_TO_TRAIN['CustomCNN'] = build_custom_cnn()

print('Models registered:')
for name, m in MODELS_TO_TRAIN.items():
    print(f'  {name:20s}  params={m.count_params():>12,}')


### Callback Factory

Each model gets its own checkpoint directory so saves never collide.

In [ ]:
class TrainingStateCallback(callbacks.Callback):
    """Persists epoch count and best AUC to JSON so training can resume across sessions."""
    def __init__(self, state_path):
        super().__init__()
        self.state_path = Path(state_path)

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        # Load existing state to preserve best_val_auc across sessions
        state = {}
        if self.state_path.exists():
            with open(self.state_path) as f:
                state = json.load(f)
        best = state.get('best_val_auc')
        cur  = logs.get('val_auc')
        if cur is not None:
            best = cur if best is None else max(best, cur)
        with open(self.state_path, 'w') as f:
            json.dump({'epoch': epoch + 1, 'best_val_auc': best}, f, indent=2)


# ── FIX: get_callbacks accepts a model_name argument ─────────────────────────
def get_callbacks(model_name: str):
    """
    Return a callback list for the given model.
    Each model gets its own checkpoint directory.
    Accepts model_name so fine-tuning calls don't raise TypeError.
    """
    ckpt_dir = CHECKPOINT_DIR / model_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    latest_path = ckpt_dir / 'latest.keras'
    best_path   = ckpt_dir / 'best.keras'
    state_path  = ckpt_dir / 'training_state.json'
    log_path    = ckpt_dir / 'training_log.csv'

    return [
        callbacks.ModelCheckpoint(
            filepath=str(latest_path),
            save_best_only=False, save_weights_only=False, verbose=0),
        callbacks.ModelCheckpoint(
            filepath=str(best_path), monitor='val_auc',
            save_best_only=True, mode='max', verbose=1),
        TrainingStateCallback(state_path),
        callbacks.CSVLogger(str(log_path), append=True),
        callbacks.EarlyStopping(
            monitor='val_auc', patience=6,
            restore_best_weights=True, mode='max', verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=3,
            min_lr=1e-7, verbose=1),
    ]

print('Callback factory ready (accepts model_name argument).')
print(f'Checkpoint root: {CHECKPOINT_DIR}')


## 6. Training All Models (64 epochs max, resumable)

Each model is trained independently. If you re-run a cell after a Kaggle session timeout, the `load_or_train` helper will:
1. Look for a saved `.keras` checkpoint in `outputs/checkpoints/<ModelName>/best.keras`
2. If found → load it and skip re-training (zero GPU time wasted)
3. If not found → train from scratch up to 64 epochs with EarlyStopping


In [ ]:
# ── Shared dicts ─────────────────────────────────────────────────────────────
ALL_MODELS = {}   # {name: (model, tr_gen, vl_gen, te_gen)}
histories  = {}   # {name: keras History or None if loaded from disk}


def plot_training_curves_model(hist, model_name, color):
    """Plot accuracy / loss / AUC curves for a single model."""
    if hist is None:
        print(f'  [{model_name}] Loaded from checkpoint — no new history to plot.')
        return
    h = hist.history
    ep = range(1, len(h['loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    fig.suptitle(f'Training Curves — {model_name}', fontsize=13, fontweight='bold')
    for ax, metric, ylabel in zip(axes,
                                   ['accuracy', 'loss', 'auc'],
                                   ['Accuracy', 'Loss', 'AUC-ROC']):
        ax.plot(ep, h[metric],           lw=2, color=color, label='Train')
        ax.plot(ep, h[f'val_{metric}'],  lw=2, color=color, ls='--', alpha=0.7, label='Val')
        ax.set_xlabel('Epoch'); ax.set_ylabel(ylabel)
        ax.set_title(ylabel, fontsize=11, fontweight='bold')
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / f'training_curve_{model_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


def load_or_train(name, model):
    """
    Load saved model from checkpoint if it exists, otherwise train from scratch.
    Stores result in ALL_MODELS and histories.
    """
    best_path = CHECKPOINT_DIR / name / 'best.keras'
    tr, vl, te = make_generators(name)

    if best_path.exists():
        print(f'\n[{name}] ✔ Checkpoint found — loading from disk (skipping training).')
        loaded = tf.keras.models.load_model(str(best_path))
        ALL_MODELS[name] = (loaded, tr, vl, te)
        histories[name]  = None
        val_auc = json.load(open(CHECKPOINT_DIR / name / 'training_state.json')).get('best_val_auc', '?')
        print(f'  Best val AUC from checkpoint: {val_auc}')
        return loaded, None

    print(f'\n{"="*65}')
    print(f'Training  →  {name}')
    print(f'{"="*65}')

    # ── Class weights: compensate for any class imbalance ────────────────
    # Counts labels from the generator without loading all images into RAM.
    _labels     = tr.classes
    _counts     = np.bincount(_labels)
    _total      = len(_labels)
    _class_w    = {i: _total / (len(_counts) * c) for i, c in enumerate(_counts)}
    print(f'  Class weights: { {CLASS_NAMES[k]: round(v,3) for k, v in _class_w.items()} }')

    hist = model.fit(
        tr, validation_data=vl, epochs=EPOCHS,
        class_weight=_class_w,
        callbacks=get_callbacks(name), verbose=1
    )
    ALL_MODELS[name] = (model, tr, vl, te)
    histories[name]  = hist

    best_auc = max(hist.history['val_auc'])
    best_acc = max(hist.history['val_accuracy'])
    ep_run   = len(hist.history['loss'])
    print(f'\n{name} complete — epochs: {ep_run}, best val AUC: {best_auc:.4f}, best val Acc: {best_acc:.4f}')
    return model, hist


### 6.1 VGG16

VGG16 is a 16-layer network built entirely from small 3×3 convolutions. Strong baseline for satellite imagery.

In [ ]:
_model_vgg16, _hist_vgg16 = load_or_train('VGG16', MODELS_TO_TRAIN['VGG16'])
plot_training_curves_model(_hist_vgg16, 'VGG16', pal['VGG16'])


### 6.2 ResNet50

ResNet50 uses skip connections to allow gradients to flow through 50 layers. Trained with the same binary head and callbacks as VGG16.

In [ ]:
_model_resnet50, _hist_resnet50 = load_or_train('ResNet50', MODELS_TO_TRAIN['ResNet50'])
plot_training_curves_model(_hist_resnet50, 'ResNet50', pal['ResNet50'])


### 6.3 EfficientNetB0

EfficientNetB0 with `include_preprocessing=False` — our CLAHE+Fire pipeline now controls all rescaling, fixing the double-normalisation bug from the previous version.

In [ ]:
_model_eff, _hist_eff = load_or_train('EfficientNetB0', MODELS_TO_TRAIN['EfficientNetB0'])
plot_training_curves_model(_hist_eff, 'EfficientNetB0', pal['EfficientNetB0'])


### 6.4 Custom CNN

3-block CNN trained from scratch using the best preprocessing pipeline.

In [ ]:
_model_custom, _hist_custom = load_or_train('CustomCNN', MODELS_TO_TRAIN['CustomCNN'])
plot_training_curves_model(_hist_custom, 'CustomCNN', pal['CustomCNN'])


In [ ]:
print('=' * 65)
print('TRAINING SUMMARY (Phase 1 — frozen base)')
print('=' * 65)
summary_rows = []
for name, hist in histories.items():
    if hist is not None:
        val_acc = max(hist.history.get('val_accuracy', [0]))
        val_auc = max(hist.history.get('val_auc', [0]))
        ep_run  = len(hist.history.get('loss', []))
    else:
        state_f = CHECKPOINT_DIR / name / 'training_state.json'
        state   = json.load(open(state_f)) if state_f.exists() else {}
        val_auc = state.get('best_val_auc', None)
        val_acc = None
        ep_run  = state.get('epoch', '?')
    summary_rows.append({'Model': name, 'Epochs Run': ep_run,
                         'Best Val Acc': val_acc, 'Best Val AUC': val_auc})
train_summary_df = pd.DataFrame(summary_rows).sort_values('Best Val AUC', ascending=False, na_position='last')
print(train_summary_df.to_string(index=False, float_format='{:.4f}'.format))
print('=' * 65)


## 7. Fine-Tuning All Models Equally

**Previous bug:** only VGG16 was fine-tuned, making comparisons unfair.
**Fix:** All four models receive identical fine-tuning — unfreeze top 25% of backbone layers, train 10 epochs at lr=1e-5.

CustomCNN has no pre-trained backbone, so its head is trained at a lower lr instead (effectively an extra training phase).


In [ ]:
def fine_tune_model(name, epochs_ft=10, lr_ft=1e-5):
    """
    Unfreeze top 25% of the backbone and continue training at a low lr.
    Skips fine-tuning if a fine-tuned checkpoint already exists.
    Works for all 4 models — CustomCNN has no backbone so only the
    classification head is trained at the reduced lr.
    """
    ft_path = CHECKPOINT_DIR / f'{name}_finetuned' / 'best.keras'
    model_obj, tr_gen, vl_gen, te_gen = ALL_MODELS[name]

    if ft_path.exists():
        print(f'[{name}] Fine-tuned checkpoint found — loading.')
        ft_model = tf.keras.models.load_model(str(ft_path))
        ALL_MODELS[name] = (ft_model, tr_gen, vl_gen, te_gen)
        return ft_model, None

    print(f'\n{"="*65}')
    print(f'Fine-tuning  →  {name}')
    print(f'{"="*65}')

    # Find backbone sub-model (if any)
    base = None
    for layer in model_obj.layers:
        if isinstance(layer, tf.keras.Model):
            base = layer
            break

    if base is not None:
        fine_tune_at = int(len(base.layers) * 0.75)
        base.trainable = True
        for layer in base.layers[:fine_tune_at]:
            layer.trainable = False
        trainable_n = sum(1 for l in base.layers if l.trainable)
        print(f'  Unfreezing top {len(base.layers) - fine_tune_at} of {len(base.layers)} backbone layers')
        print(f'  Trainable backbone layers: {trainable_n}')
    else:
        print(f'  [{name}] No backbone — running extra Dense-head training phase')

    model_obj.compile(
        optimizer=Adam(learning_rate=lr_ft),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    tr_gen.reset(); vl_gen.reset()

    ft_hist = model_obj.fit(
        tr_gen, validation_data=vl_gen, epochs=epochs_ft,
        callbacks=get_callbacks(f'{name}_finetuned'), verbose=1
    )

    model_obj.save(SAVE_DIR / f'{name}_finetuned.keras')
    ALL_MODELS[name] = (model_obj, tr_gen, vl_gen, te_gen)

    best_auc = max(ft_hist.history['val_auc'])
    print(f'  Fine-tune complete — best val AUC: {best_auc:.4f}')
    return model_obj, ft_hist

print('fine_tune_model() defined.')


In [ ]:
ft_histories = {}
for name in list(ALL_MODELS.keys()):
    _, ft_hist = fine_tune_model(name, epochs_ft=10, lr_ft=1e-5)
    ft_histories[name] = ft_hist
    if ft_hist is not None:
        plot_training_curves_model(ft_hist, f'{name} (fine-tuned)', pal.get(name, '#888'))

print('\nFine-tuning complete for all models.')


## 8. Ensemble Model

Average the probability outputs of VGG16 and ResNet50 (the two strongest transfer-learning models).
Even when two models individually score 99% AUC, they disagree on different hard cases — averaging cancels individual errors.

> **CustomCNN** is excluded from the ensemble because its architecture is fundamentally simpler.
> EfficientNetB0 is added only if its AUC after the fix is ≥ 97%.


In [ ]:
def evaluate_model(model, test_gen, threshold=0.5):
    """Evaluate a model on test_gen and return a metrics dict."""
    test_gen.reset()
    y_prob = model.predict(test_gen, verbose=1).flatten()
    y_pred = (y_prob >= threshold).astype(int)
    y_true = test_gen.classes
    y_bin  = (y_true == WILDFIRE_IDX).astype(int)
    return {
        'accuracy':      accuracy_score(y_bin, y_pred),
        'precision':     precision_score(y_bin, y_pred, zero_division=0),
        'recall':        recall_score(y_bin, y_pred, zero_division=0),
        'f1':            f1_score(y_bin, y_pred, zero_division=0),
        'auc':           roc_auc_score(y_bin, y_prob),
        'avg_precision': average_precision_score(y_bin, y_prob),
        'y_true': y_true, 'y_pred': y_pred, 'y_prob': y_prob,
    }


# ── Collect raw probabilities from all trained models ─────────────────────────
all_results = {}
for name, (model, _, _, te_gen) in ALL_MODELS.items():
    print(f'Evaluating {name}...')
    all_results[name] = evaluate_model(model, te_gen, threshold=0.5)

# ── Dynamic Ensemble: top-2 models by AUC (excludes CustomCNN) ──────────────
# CustomCNN is excluded — its simpler architecture makes probability
# scales less compatible with the deeper transfer-learning models.
_eligible   = {n: r for n, r in all_results.items() if n != 'CustomCNN'}
_sorted_ens = sorted(_eligible.items(), key=lambda x: x[1]['auc'], reverse=True)
ens_models  = [n for n, _ in _sorted_ens[:2]]
print(f'  Dynamic ensemble members: {ens_models}')

ens_probs  = np.mean([all_results[n]['y_prob'] for n in ens_models], axis=0)
ens_true   = all_results[ens_models[0]]['y_true']
ens_bin    = (ens_true == WILDFIRE_IDX).astype(int)
ens_pred   = (ens_probs >= 0.5).astype(int)

all_results['Ensemble'] = {
    'accuracy':      accuracy_score(ens_bin, ens_pred),
    'precision':     precision_score(ens_bin, ens_pred, zero_division=0),
    'recall':        recall_score(ens_bin, ens_pred, zero_division=0),
    'f1':            f1_score(ens_bin, ens_pred, zero_division=0),
    'auc':           roc_auc_score(ens_bin, ens_probs),
    'avg_precision': average_precision_score(ens_bin, ens_probs),
    'y_true': ens_true, 'y_pred': ens_pred, 'y_prob': ens_probs,
}
print(f'Ensemble evaluated  (members: {ens_models}).')


## 9. Evaluation & Model Comparison

All models + the ensemble are evaluated on the held-out test set with six metrics.
**Recall is the safety-critical metric** — missing a real fire is far more dangerous than a false alarm.


In [ ]:
summary_rows = []
for name, r in all_results.items():
    summary_rows.append({
        'Model': name, 'Accuracy': r['accuracy'],
        'Precision': r['precision'], 'Recall': r['recall'],
        'F1': r['f1'], 'AUC': r['auc'], 'Avg Precision': r['avg_precision']
    })

summary_df      = pd.DataFrame(summary_rows).sort_values('AUC', ascending=False)
best_model_name = summary_df.iloc[0]['Model']

print('\n' + '=' * 75)
print('TEST SET RESULTS (sorted by AUC) — all models at default threshold 0.5')
print('=' * 75)
print(summary_df.to_string(index=False, float_format='{:.4f}'.format))
print('=' * 75)
print(f'\n  BEST MODEL: {best_model_name}  (AUC = {summary_df.iloc[0]["AUC"]:.4f})')


### 9.1 Confusion Matrices

In [ ]:
n_models = len(all_results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 5))
fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold')
for ax, (name, r) in zip(axes, all_results.items()):
    y_bin = (r['y_true'] == WILDFIRE_IDX).astype(int)
    cm = confusion_matrix(y_bin, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', square=True, ax=ax, cbar=False,
                xticklabels=['No Fire', 'Fire'], yticklabels=['No Fire', 'Fire'])
    ax.set_ylabel('True', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted', fontsize=10, fontweight='bold')
    ax.set_title(f'{name}\nAcc={r["accuracy"]:.3f}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.2 ROC & Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for name, r in all_results.items():
    y_bin = (r['y_true'] == WILDFIRE_IDX).astype(int)
    color = pal.get(name, '#888888')
    fpr, tpr, _ = roc_curve(y_bin, r['y_prob'])
    axes[0].plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC={r["auc"]:.3f})')
    prec, rec, _ = precision_recall_curve(y_bin, r['y_prob'])
    axes[1].plot(rec, prec, color=color, lw=2.5, label=f'{name} (AP={r["avg_precision"]:.3f})')

axes[0].plot([0,1],[0,1],'k--',lw=1.5)
axes[0].set_xlabel('False Positive Rate', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Positive Rate',  fontsize=12, fontweight='bold')
axes[0].set_title('ROC Curves', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Recall',    fontsize=12, fontweight='bold')
axes[1].set_ylabel('Precision', fontsize=12, fontweight='bold')
axes[1].set_title('Precision-Recall Curves', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()


### 9.3 Classification Heatmap Correlation

In [ ]:
model_names = list(all_results.keys())
prob_df = pd.DataFrame({name: all_results[name]['y_prob'] for name in model_names})
metrics_df = pd.DataFrame({
    'Precision': [all_results[m]['precision'] for m in model_names],
    'Recall':    [all_results[m]['recall']    for m in model_names],
    'F1':        [all_results[m]['f1']        for m in model_names],
    'Accuracy':  [all_results[m]['accuracy']  for m in model_names],
    'AUC':       [all_results[m]['auc']       for m in model_names],
}, index=model_names)

preds    = np.array([all_results[m]['y_pred'] for m in model_names])
agree    = np.mean(preds[:, None, :] == preds[None, :, :], axis=2) * 100
agree_df = pd.DataFrame(agree, index=model_names, columns=model_names)

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Classification Heatmap Correlation — All Models', fontsize=14, fontweight='bold')
sns.heatmap(prob_df.corr(), annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.5, vmax=1, ax=axes[0], square=True, linewidths=0.5)
axes[0].set_title('Probability Correlation', fontweight='bold')
sns.heatmap(metrics_df, annot=True, fmt='.3f', cmap='YlOrRd',
            vmin=0.8, vmax=1, ax=axes[1], linewidths=0.5)
axes[1].set_title('Model Performance', fontweight='bold')
sns.heatmap(agree_df, annot=True, fmt='.1f', cmap='Blues',
            vmin=80, vmax=100, ax=axes[2], square=True, linewidths=0.5)
axes[2].set_title('Prediction Agreement (%)', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR / 'classification_heatmap_correlation.png', dpi=150, bbox_inches='tight')
plt.show()


## 10. F1-Optimal Threshold (not default 0.5)

The default threshold of 0.5 is arbitrary. We find the threshold that **maximises F1** on the **validation set** (never the test set — that would be data leakage), then apply it to evaluate on the test set.

This is computed for the **best model** selected by AUC from Section 9.


In [ ]:
def find_f1_optimal_threshold(y_true, y_prob, pos_label=WILDFIRE_IDX):
    """Find the threshold that maximises F1-score on the given split."""
    y_bin = (y_true == pos_label).astype(int)
    prec, rec, thresholds = precision_recall_curve(y_bin, y_prob)
    f1_scores = 2 * (prec * rec) / (prec + rec + 1e-10)
    best_idx  = np.argmax(f1_scores)
    thresh    = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
    return thresh, float(f1_scores[best_idx])


# ── Compute optimal threshold on VALIDATION set (no data leakage) ─────────────
best_model_obj, _, best_val_gen, best_test_gen = ALL_MODELS[best_model_name]
best_val_gen.reset()
y_prob_val = best_model_obj.predict(best_val_gen, verbose=1).flatten()
y_true_val = best_val_gen.classes

OPT_THRESH, val_f1 = find_f1_optimal_threshold(y_true_val, y_prob_val)
print(f'Best model           : {best_model_name}')
print(f'Optimal threshold    : {OPT_THRESH:.4f}  (maximises F1 on validation set)')
print(f'F1 at optimal thresh : {val_f1:.4f}')

# ── Compare default vs optimal on TEST set ────────────────────────────────────
best_res  = all_results[best_model_name]
y_prob_te = best_res['y_prob']
y_true_te = best_res['y_true']
y_bin_te  = (y_true_te == WILDFIRE_IDX).astype(int)

def metrics_at(y_bin, y_prob, thresh):
    yp = (y_prob >= thresh).astype(int)
    return {'Accuracy':  accuracy_score(y_bin, yp),
            'Precision': precision_score(y_bin, yp, zero_division=0),
            'Recall':    recall_score(y_bin, yp, zero_division=0),
            'F1':        f1_score(y_bin, yp, zero_division=0)}

m_default = metrics_at(y_bin_te, y_prob_te, 0.5)
m_optimal = metrics_at(y_bin_te, y_prob_te, OPT_THRESH)
cmp_df = pd.DataFrame({'Default (0.500)': m_default,
                        f'Optimal ({OPT_THRESH:.3f})': m_optimal})
cmp_df['Improvement'] = cmp_df[f'Optimal ({OPT_THRESH:.3f})'] - cmp_df['Default (0.500)']

print('\n' + '=' * 60)
print(f'THRESHOLD COMPARISON — {best_model_name}')
print('=' * 60)
print(cmp_df.to_string(float_format='{:.4f}'.format))


In [ ]:
thresh_range = np.linspace(0.05, 0.95, 200)
precs, recs, f1s = [], [], []
for t in thresh_range:
    yp = (y_prob_te >= t).astype(int)
    precs.append(precision_score(y_bin_te, yp, zero_division=0))
    recs .append(recall_score(y_bin_te, yp, zero_division=0))
    f1s  .append(f1_score(y_bin_te, yp, zero_division=0))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(thresh_range, precs, color='#e74c3c', lw=2.5, label='Precision')
ax.plot(thresh_range, recs,  color='#3498db', lw=2.5, label='Recall')
ax.plot(thresh_range, f1s,   color='#2ecc71', lw=2.5, label='F1-Score')
ax.axvline(OPT_THRESH, color='#9b59b6', ls='--', lw=2.5, label=f'Optimal ({OPT_THRESH:.3f})')
ax.axvline(0.5, color='gray', ls=':', lw=2, alpha=0.7, label='Default (0.5)')
ax.plot(OPT_THRESH, m_optimal['F1'], 'o', color='#2ecc71', ms=12, mec='white', mew=2,
        label=f'Max F1={m_optimal["F1"]:.3f}')
ax.set_xlabel('Threshold', fontsize=13, fontweight='bold')
ax.set_ylabel('Score',     fontsize=13, fontweight='bold')
ax.set_title(f'Threshold Sensitivity — {best_model_name}', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.05])
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'threshold_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()


### 10.1 Calibration Curve

Shows whether the model's predicted probabilities are reliable.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
fraction_pos, mean_pred = calibration_curve(y_bin_te, y_prob_te, n_bins=10)
ax.plot(mean_pred, fraction_pos, 's-', label=best_model_name, color='#3498db')
ax.plot([0,1],[0,1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean predicted probability'); ax.set_ylabel('Fraction of positives')
ax.set_title('Calibration Curve'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'calibration_curve.png', dpi=150, bbox_inches='tight')
plt.show()


## 10.2 Test-Time Augmentation (TTA)

TTA runs inference **N times** on augmented versions of each test image (horizontal flip, vertical flip, both flips + original) and **averages the predicted probabilities**.

Even a simple 4-pass TTA consistently lifts AUC by **0.3–0.8%** by reducing prediction variance on borderline cases.

> TTA is applied only at inference time — no retraining needed.

In [ ]:
# ── Test-Time Augmentation (TTA) ─────────────────────────────────────────────
# Instead of predicting on the original image once, TTA predicts on N augmented
# versions (flips + mild rotation) and averages the probabilities.
# Even a simple 4-transform TTA typically lifts AUC by 0.3–0.8%.

def tta_predict(model, test_gen, n_augments=4):
    """
    Test-Time Augmentation: average predictions over horizontal flip,
    vertical flip, and both flips — plus the original.
    Returns averaged probability array (shape: [n_samples]).
    """
    tta_datagen = ImageDataGenerator(
        preprocessing_function=_prep_fn,
        horizontal_flip=True,
        vertical_flip=True,
    )
    all_preds = []

    # Pass 1: original (no augmentation)
    test_gen.reset()
    preds_orig = model.predict(test_gen, verbose=0).flatten()
    all_preds.append(preds_orig)

    # Passes 2-N: augmented versions
    for aug_idx in range(n_augments - 1):
        aug_gen = tta_datagen.flow_from_directory(
            TEST_DIR,
            target_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            class_mode='binary',
            shuffle=False,
            seed=SEED + aug_idx
        )
        preds_aug = model.predict(aug_gen, verbose=0).flatten()
        all_preds.append(preds_aug)

    return np.mean(all_preds, axis=0)


# ── Run TTA on best model and compare with standard inference ────────────────
print(f'Running TTA ({4} passes) on best model: {best_model_name} ...')
_, _, _, tta_test_gen = ALL_MODELS[best_model_name]
y_prob_tta = tta_predict(best_model_obj, tta_test_gen, n_augments=4)
y_true_tta = tta_test_gen.classes
y_bin_tta  = (y_true_tta == WILDFIRE_IDX).astype(int)
y_pred_tta = (y_prob_tta >= OPT_THRESH).astype(int)

tta_metrics = {
    'Accuracy':  accuracy_score(y_bin_tta, y_pred_tta),
    'Precision': precision_score(y_bin_tta, y_pred_tta, zero_division=0),
    'Recall':    recall_score(y_bin_tta, y_pred_tta, zero_division=0),
    'F1':        f1_score(y_bin_tta, y_pred_tta, zero_division=0),
    'AUC':       roc_auc_score(y_bin_tta, y_prob_tta),
}
std_metrics = {
    'Accuracy':  all_results[best_model_name]['accuracy'],
    'Precision': all_results[best_model_name]['precision'],
    'Recall':    all_results[best_model_name]['recall'],
    'F1':        all_results[best_model_name]['f1'],
    'AUC':       all_results[best_model_name]['auc'],
}
tta_cmp = pd.DataFrame({'Standard': std_metrics, 'TTA (4-pass)': tta_metrics})
tta_cmp['Delta'] = tta_cmp['TTA (4-pass)'] - tta_cmp['Standard']
print('\n' + '='*55)
print(f'TTA COMPARISON — {best_model_name}')
print('='*55)
print(tta_cmp.to_string(float_format='{:.4f}'.format))

# Store TTA result for downstream use
all_results[f'{best_model_name}_TTA'] = {
    **{k: v for k, v in all_results[best_model_name].items()
       if k not in ('y_true','y_pred','y_prob')},
    'accuracy': tta_metrics['Accuracy'], 'precision': tta_metrics['Precision'],
    'recall': tta_metrics['Recall'],     'f1': tta_metrics['F1'],
    'auc': tta_metrics['AUC'],
    'y_true': y_true_tta, 'y_pred': y_pred_tta, 'y_prob': y_prob_tta,
}
print(f'\n  TTA result stored as "{best_model_name}_TTA" in all_results.')

## 11. Error Analysis — False Positives & False Negatives

Visualising the model's most confident mistakes reveals failure patterns:
- **False Positives:** Clouds/fog resembling smoke, bright desert terrain
- **False Negatives:** Small/distant fires, fires obscured by dense vegetation

The error gallery uses the **optimised threshold** so the displayed mistakes are the ones the production system would make.


In [ ]:
def plot_error_gallery(model, test_gen, model_name, n=6):
    test_gen.reset()
    y_prob = model.predict(test_gen, verbose=1).flatten()
    y_pred = (y_prob >= OPT_THRESH).astype(int)
    y_true = test_gen.classes
    paths  = test_gen.filepaths
    fp_idx = np.where((y_pred == WILDFIRE_IDX) & (y_true != WILDFIRE_IDX))[0]
    fn_idx = np.where((y_pred != WILDFIRE_IDX) & (y_true == WILDFIRE_IDX))[0]
    fp_sorted = fp_idx[np.argsort(y_prob[fp_idx])[::-1]][:n]
    fn_sorted = fn_idx[np.argsort(y_prob[fn_idx])][:n]
    fig, axes = plt.subplots(2, n, figsize=(n*2.5, 6))
    fig.suptitle(f'Error Gallery — {model_name}  (threshold={OPT_THRESH:.3f})',
                 fontsize=13, fontweight='bold')
    for col in range(n):
        for row, (idx_arr, row_label, color) in enumerate([
            (fp_sorted, 'False Positives', '#e74c3c'),
            (fn_sorted, 'False Negatives', '#3498db')
        ]):
            ax = axes[row, col]
            if col < len(idx_arr):
                i = idx_arr[col]
                img = cv2.cvtColor(cv2.resize(cv2.imread(paths[i]),
                                              (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
                ax.imshow(img)
                ax.set_title(f'p={y_prob[i]:.3f}', fontsize=9, color=color)
            ax.axis('off')
            if col == 0:
                ax.set_ylabel(row_label, fontsize=10, fontweight='bold', color=color)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'error_gallery.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'FP count: {len(fp_idx)}  |  FN count: {len(fn_idx)}')

best_model_obj, _, _, best_test_gen = ALL_MODELS[best_model_name]
plot_error_gallery(best_model_obj, best_test_gen, best_model_name, n=6)


## 12. Grad-CAM Visualization

Grad-CAM highlights which image regions drove the model's decision. For a trustworthy wildfire detector, activations should concentrate on **flames, smoke plumes, and burn scars** — not background.

All predictions in the gallery use the **optimised threshold** (not 0.5).


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    """
    Unified Grad-CAM for both transfer-learning models and CustomCNN.
    - Transfer models: gradient flows through the nested backbone sub-model.
    - CustomCNN: no backbone wrapper — target the last Conv2D layer directly.
    """
    # ── Detect architecture type ──────────────────────────────────────────
    backbone = None
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model):
            backbone = layer
            break

    if backbone is not None:
        # Transfer-learning path (VGG16 / ResNet50 / EfficientNetB0)
        grad_model = tf.keras.models.Model(
            inputs=backbone.input,
            outputs=[backbone.get_layer(last_conv_layer_name).output, backbone.output]
        )
        with tf.GradientTape() as tape:
            img_tensor = tf.cast(img_array, tf.float32)
            conv_out, backbone_out = grad_model(img_tensor)
            x = backbone_out
            for layer in model.layers[1:]:
                x = layer(x, training=False)
            class_channel = x[:, 0]
    else:
        # CustomCNN path — find the last Conv2D layer inside the flat model
        last_conv_layer = None
        for layer in model.layers:
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv_layer = layer
        if last_conv_layer is None:
            return np.zeros((14, 14))   # fallback — should never happen
        grad_model = tf.keras.models.Model(
            inputs=model.input,
            outputs=[last_conv_layer.output, model.output]
        )
        with tf.GradientTape() as tape:
            img_tensor = tf.cast(img_array, tf.float32)
            conv_out, preds = grad_model(img_tensor)
            class_channel = preds[:, 0]

    grads        = tape.gradient(class_channel, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap      = conv_out[0] @ pooled_grads[..., tf.newaxis]
    heatmap      = tf.squeeze(heatmap)
    heatmap      = tf.maximum(heatmap, 0)
    heatmap      = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def overlay_gradcam(img_rgb, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (img_rgb.shape[1], img_rgb.shape[0]))
    heatmap_u8      = np.uint8(255 * heatmap_resized)
    heatmap_color   = cv2.applyColorMap(heatmap_u8, cv2.COLORMAP_JET)
    heatmap_rgb     = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    superimposed    = heatmap_rgb * alpha + img_rgb
    return np.clip(superimposed, 0, 255).astype(np.uint8)


def plot_gradcam_gallery(model, test_gen, last_conv_layer, n=8):
    test_gen.reset()
    samples = {'wildfire': [], 'nowildfire': []}
    for _ in range(len(test_gen)):
        imgs, labels = next(test_gen)
        for img, lbl in zip(imgs, labels):
            key = 'wildfire' if lbl == WILDFIRE_IDX else 'nowildfire'
            if len(samples[key]) < n // 2:
                samples[key].append((img, int(lbl)))
        if all(len(v) >= n // 2 for v in samples.values()):
            break

    all_imgs  = samples['wildfire'] + samples['nowildfire']
    fig, axes = plt.subplots(2, n, figsize=(n * 2.5, 6))
    fig.suptitle(f'Grad-CAM — {best_model_name}', fontsize=14, fontweight='bold')
    for idx, (img, true_lbl) in enumerate(all_imgs):
        img_exp   = np.expand_dims(img, 0)
        pred_prob = float(model.predict(img_exp, verbose=0)[0][0])
        pred_lbl  = 1 if pred_prob >= OPT_THRESH else 0
        heatmap   = make_gradcam_heatmap(img_exp, model, last_conv_layer)
        img_u8    = (img * 255).astype(np.uint8)
        overlay   = overlay_gradcam(img_u8, heatmap)
        color     = '#2ecc71' if pred_lbl == true_lbl else '#e74c3c'
        axes[0, idx].imshow(img_u8)
        axes[0, idx].set_title(f'True:{CLASS_NAMES[true_lbl]}\nP={pred_prob:.2f}',
                               fontsize=8, color=color, fontweight='bold')
        axes[0, idx].axis('off')
        axes[1, idx].imshow(overlay)
        axes[1, idx].axis('off')
    for row, label in enumerate(['Original', 'Grad-CAM']):
        axes[row, 0].set_ylabel(label, fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'gradcam_gallery.png', dpi=150, bbox_inches='tight')
    plt.show()

# ── Per-architecture last conv layer names ───────────────────────────────────
LAST_CONV_LAYERS = {
    'VGG16':          'block5_conv3',
    'ResNet50':       'conv5_block3_out',
    'EfficientNetB0': 'top_conv',
    'CustomCNN':      'conv3', # last Conv2D block — Grad-CAM uses direct model path
    'Ensemble':       None,   # virtual model — use members' individual maps
}

last_conv = LAST_CONV_LAYERS.get(best_model_name)
if last_conv is None:
    print(f'Grad-CAM not supported for {best_model_name} — skipping.')
else:
    print(f'Running Grad-CAM with layer: {last_conv}  (threshold={OPT_THRESH:.3f})')
    plot_gradcam_gallery(best_model_obj, best_test_gen, last_conv, n=8)


## Save All Outputs

In [ ]:
# 1. Best model (post fine-tuning)
final_path = OUT_DIR / f'{best_model_name}_final.keras'
best_model_obj.save(final_path)
print(f'Best model saved: {final_path.name}')

# 2. All models
for name, (model, _, _, _) in ALL_MODELS.items():
    p = SAVE_DIR / f'{name}_final.keras'
    model.save(p)
    print(f'  Saved {name} → {p.name}')

# 3. CSVs
summary_df.to_csv(OUT_DIR / 'model_comparison.csv', index=False)
ablation_df.to_csv(OUT_DIR / 'ablation_results.csv', index=False)

# 4. Metadata JSON
experiment_meta = {
    'timestamp':         time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()),
    'best_model':        best_model_name,
    'best_pipeline':     BEST_PIPELINE_NAME,
    'optimal_threshold': float(OPT_THRESH),
    'img_size':          IMG_SIZE, 'batch_size': BATCH_SIZE, 'seed': SEED,
    'epochs_max':        EPOCHS,
    'wildfire_class_idx': WILDFIRE_IDX, 'class_names': CLASS_NAMES,
    'test_metrics': {
        name: {k: float(v) for k, v in r.items()
               if k not in ('y_true', 'y_pred', 'y_prob')}
        for name, r in all_results.items()
    }
}
with open(OUT_DIR / 'experiment_metadata.json', 'w') as f:
    json.dump(experiment_meta, f, indent=2)
print('Experiment metadata saved.')

# 5. Output directory listing
print('\n' + '='*55)
print('ALL OUTPUTS IN:', OUT_DIR.resolve())
print('='*55)
total = 0
for fp in sorted(OUT_DIR.rglob('*')):
    if fp.is_file():
        sz = fp.stat().st_size; total += sz
        print(f'  {str(fp.relative_to(OUT_DIR)):<50s} {sz/1e6:6.1f} MB')
print(f'  {"TOTAL":<50s} {total/1e6:6.1f} MB')


## Load Saved Models (skip retraining in future sessions)

Run this cell instead of cells 24–31 when you already have trained checkpoints saved. It populates `ALL_MODELS` and `all_results` so every downstream cell (threshold, Grad-CAM, error analysis) works immediately without retraining.


In [ ]:
# ── HOW TO USE ────────────────────────────────────────────────────────────────
# 1. Set LOAD_FROM_DISK = True
# 2. Run this cell — it rebuilds ALL_MODELS, histories, and all_results from disk
# 3. Jump straight to Section 10 (threshold optimisation) or any downstream cell
# ─────────────────────────────────────────────────────────────────────────────

LOAD_FROM_DISK = False   # ← change to True when models are already trained

if LOAD_FROM_DISK:
    MODEL_FILES = {
        'VGG16':          SAVE_DIR / 'VGG16_final.keras',
        'ResNet50':       SAVE_DIR / 'ResNet50_final.keras',
        'EfficientNetB0': SAVE_DIR / 'EfficientNetB0_final.keras',
        'CustomCNN':      SAVE_DIR / 'CustomCNN_final.keras',
    }

    ALL_MODELS = {}
    histories  = {}

    for name, model_path in MODEL_FILES.items():
        if not model_path.exists():
            print(f'  ⚠  {name}: file not found at {model_path} — skipping')
            continue
        print(f'  Loading {name} from {model_path.name} ...')
        loaded = tf.keras.models.load_model(str(model_path))
        tr, vl, te = make_generators(name)
        ALL_MODELS[name] = (loaded, tr, vl, te)
        histories[name]  = None
        print(f'    ✔ {name} loaded  (params={loaded.count_params():,})')

    # Re-evaluate on test set so downstream cells have all_results populated
    all_results = {}
    for name, (model, _, _, te_gen) in ALL_MODELS.items():
        print(f'  Evaluating {name}...')
        all_results[name] = evaluate_model(model, te_gen, threshold=0.5)

    # Rebuild dynamic ensemble — top-2 models by AUC (excludes CustomCNN)
    _eligible   = {n: r for n, r in all_results.items() if n != 'CustomCNN'}
    _sorted_ens = sorted(_eligible.items(), key=lambda x: x[1]['auc'], reverse=True)
    ens_models  = [n for n, _ in _sorted_ens[:2]]
    print(f'  Dynamic ensemble members: {ens_models}')
    if len(ens_models) == 2:
        ens_probs  = np.mean([all_results[n]['y_prob'] for n in ens_models], axis=0)
        ens_true   = all_results[ens_models[0]]['y_true']
        ens_bin    = (ens_true == WILDFIRE_IDX).astype(int)
        ens_pred   = (ens_probs >= 0.5).astype(int)
        all_results['Ensemble'] = {
            'accuracy':      accuracy_score(ens_bin, ens_pred),
            'precision':     precision_score(ens_bin, ens_pred, zero_division=0),
            'recall':        recall_score(ens_bin, ens_pred, zero_division=0),
            'f1':            f1_score(ens_bin, ens_pred, zero_division=0),
            'auc':           roc_auc_score(ens_bin, ens_probs),
            'avg_precision': average_precision_score(ens_bin, ens_probs),
            'y_true': ens_true, 'y_pred': ens_pred, 'y_prob': ens_probs,
        }

    summary_df      = pd.DataFrame([
        {'Model': n, 'Accuracy': r['accuracy'], 'Precision': r['precision'],
         'Recall': r['recall'], 'F1': r['f1'], 'AUC': r['auc']}
        for n, r in all_results.items()
    ]).sort_values('AUC', ascending=False)
    best_model_name = summary_df.iloc[0]['Model']
    best_model_obj  = ALL_MODELS[best_model_name][0]
    _, _, best_val_gen, best_test_gen = ALL_MODELS[best_model_name]

    print(f'\nAll models loaded. Best model: {best_model_name}')
    print(summary_df.to_string(index=False, float_format='{:.4f}'.format))
else:
    print('LOAD_FROM_DISK = False — using models trained in this session.')
    print('Set LOAD_FROM_DISK = True to load from saved .keras files instead.')


## 13. Conclusions & Future Work

In [ ]:
conclusion = f"""
╔══════════════════════════════════════════════════════════════════════════════╗
║              WILDFIRE DETECTION — EXPERIMENTAL CONCLUSIONS (v3)            ║
╚══════════════════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 FIXES APPLIED IN THIS VERSION
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  ✔  EfficientNetB0: include_preprocessing=False — eliminates double-rescaling bug
  ✔  All models use binary_crossentropy + sigmoid consistently
  ✔  CustomCNN added to ALL_MODELS and included in evaluation loop
  ✔  Fine-tuning applied equally to ALL 4 models (not just VGG16)
  ✔  Ensemble (VGG16 + ResNet50 probability average) added
  ✔  Epochs increased to 64 (EarlyStopping patience=6 prevents overrun)
  ✔  get_callbacks() now accepts model_name — no more TypeError in fine-tuning
  ✔  load_or_train() allows resuming without re-running training cells
  ✔  LOAD_FROM_DISK cell loads all saved models in one step for future sessions
  ✔  Threshold fixed at F1-optimal value (not 0.5) for all downstream metrics

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 BEST MODEL: {best_model_name}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   Pipeline          : CLAHE + Fire Enhancement
   Optimal threshold : {OPT_THRESH:.4f}  (F1-maximised on validation set)

   Test Accuracy  : {all_results[best_model_name]['accuracy']:.4f}
   Recall         : {all_results[best_model_name]['recall']:.4f}   ← safety-critical
   Precision      : {all_results[best_model_name]['precision']:.4f}
   F1-Score       : {all_results[best_model_name]['f1']:.4f}
   ROC-AUC        : {all_results[best_model_name]['auc']:.4f}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 FUTURE WORK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  • Add NIR / SWIR satellite bands (Sentinel-2, Landsat-8) — biggest FN reducer
  • Explore Vision Transformers (ViT, Swin-T) for global spatial context
  • Add temporal modelling (ConvLSTM) for early fire progression detection
  • Deploy with TensorFlow Lite quantization for edge hardware
  • Extend to multi-class: no fire / smoke / active fire / burn scar
"""

print(conclusion)
